# Read an NCI PDQ cancer JSON file

This demo locates `data/nci_pdq/cancers`, loads one cancer JSON file with UTF-8 encoding, and inspects its pages and sections.

In [1]:
import json
from pathlib import Path


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the project root when Jupyter starts in the root or rag folder."""
    for candidate in (start, *start.parents):
        cancer_dir = candidate / "data" / "nci_pdq" / "cancers"
        if cancer_dir.is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find data/nci_pdq/cancers above the current directory."
    )


PROJECT_ROOT = find_project_root()
CANCER_DATA_DIR = PROJECT_ROOT / "data" / "nci_pdq" / "cancers"

print(f"Project root: {PROJECT_ROOT}")
print(f"Cancer data:  {CANCER_DATA_DIR}")

Project root: d:\Personal Project\RAG In Cancer Myth\rag-cancer-myth
Cancer data:  d:\Personal Project\RAG In Cancer Myth\rag-cancer-myth\data\nci_pdq\cancers


In [2]:
# Show a few available files, then select one for the demo.
cancer_files = sorted(CANCER_DATA_DIR.glob("*.json"))

if not cancer_files:
    raise FileNotFoundError(f"No JSON files found in {CANCER_DATA_DIR}")

print(f"Found {len(cancer_files)} cancer JSON files.")
for path in cancer_files[:10]:
    print(f"- {path.name}")

# Change this filename to inspect a different cancer.
json_path = CANCER_DATA_DIR / "acute_lymphoblastic_leukemia.json"
if not json_path.is_file():
    json_path = cancer_files[0]

print(f"\nSelected file: {json_path.name}")

Found 166 cancer JSON files.
- acute_lymphoblastic_leukemia.json
- acute_myeloid_leukemia.json
- adrenocortical_carcinoma.json
- adult_central_nervous_system_tumors.json
- aggressive_b_cell_non_hodgkin_lymphoma.json
- agnostic_cancer_therapies.json
- aids_related_lymphoma.json
- anal_cancer.json
- bile_duct_cancer_cholangiocarcinoma.json
- bladder_and_other_urothelial_cancers.json

Selected file: acute_lymphoblastic_leukemia.json


In [3]:
# Explicit UTF-8 preserves characters such as PDQ® and en dashes.
with json_path.open(mode="r", encoding="utf-8") as file:
    cancer_data = json.load(file)

print(f"Top-level keys: {list(cancer_data)}")
print(f"Cancer type:    {cancer_data.get('cancer_type')}")
print(f"Page count:     {cancer_data.get('page_count')}")

Top-level keys: ['schema_version', 'source', 'collection', 'cancer_type', 'generated_at', 'page_count', 'pages']
Cancer type:    Acute Lymphoblastic Leukemia
Page count:     2


In [4]:
# Inspect the first NCI page stored in this cancer file.
pages = cancer_data.get("pages", [])
if not pages:
    raise ValueError(f"The file has no page records: {json_path}")

first_page = pages[0]
print(f"Title:        {first_page.get('title')}")
print(f"Topic:        {first_page.get('topic')}")
print(f"Audience:     {first_page.get('audience')}")
print(f"Last updated: {first_page.get('last_updated')}")
print(f"URL:          {first_page.get('url')}")
print(f"Sections:     {len(first_page.get('sections', []))}")

Title:        Acute Lymphoblastic Leukemia Treatment (PDQ®)–Health Professional Version
Topic:        adult_treatment
Audience:     health_professional
Last updated: March 17, 2025
URL:          https://www.cancer.gov/types/leukemia/hp/adult-all-treatment-pdq
Sections:     45


In [5]:
# Display the first useful (non-boilerplate) section.
sections = first_page.get("sections", [])
first_section = next(
    (
        section
        for section in sections
        if not section.get("is_boilerplate", False)
        and section.get("text", "").strip()
    ),
    None,
)

if first_section is None:
    print("No non-boilerplate section was found.")
else:
    print(f"Section: {first_section.get('section')}")
    print(f"Path:    {first_section.get('section_path')}")
    print("\nText preview:")
    print(first_section['text'][:1_000])

Section: General Information About Acute Lymphoblastic Leukemia (ALL)
Path:    ['Acute Lymphoblastic Leukemia Treatment (PDQ®)–Health Professional Version', 'General Information About Acute Lymphoblastic Leukemia (ALL)']

Text preview:
ALL (also called acute lymphocytic leukemia) is an aggressive type of leukemia characterized by the presence of too many lymphoblasts or lymphocytes in the bone marrow and peripheral blood. It can spread to the lymph nodes, spleen, liver, central nervous system (CNS), testicles, and other organs. Without treatment, ALL usually progresses quickly.
Signs and symptoms of ALL may include:
- Weakness or fatigue.
- Fever or night sweats.
- Bruises or bleeds easily (i.e., bleeding gums, purplish patches in the skin, or petechiae [flat, pinpoint spots under the skin]).
- Shortness of breath.
- Unexpected weight loss or anorexia.
- Pain in the bones or joints.
- Swollen lymph nodes, particularly lymph nodes in the neck, armpit, or groin, which are usually painles

In [6]:
# Optional reusable helper for loading any cancer file by filename.
def load_cancer_json(filename: str) -> dict:
    path = CANCER_DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Cancer JSON file does not exist: {path}")
    with path.open(mode="r", encoding="utf-8") as file:
        return json.load(file)


breast_cancer_data = load_cancer_json("breast_cancer.json")
print(breast_cancer_data["cancer_type"], breast_cancer_data["page_count"])

Breast Cancer 3
